# Can `next_days.csv` be ingested by the current pipeline?

**Goal: implementability, not accuracy.** `dataset/next_days.csv` holds real NYC 311 tickets created 2026-03-10 to 03-12, i.e. *after* the corpus the system was built on. This notebook feeds them through our own functions to show that the *evolving data* requirement works:

1. schema matches what the pipeline needs
2. the pipeline's own cleaner produces usable tickets
3. no overlap with the training state (no leakage / double counting)
4. how much of the new data falls into known patterns vs new ticket classes, and how much resolution drift there is
5. a dry-run of the real ingestion (`apply_batch`) on a **scratch copy** of the state (`data/processed` is never modified)
6. idempotency, and that new classes become searchable

Run from the repo root venv (`pip install ipykernel` once, then pick the `.venv` kernel).

In [ ]:
import os, json, shutil, tempfile
from pathlib import Path
import numpy as np
import pandas as pd

if Path.cwd().name == 'EDA':
    os.chdir(Path.cwd().parent)  # notebooks run from the repo root
print('cwd:', Path.cwd())

from ticketrag.patterns import clean_tickets, build_pattern_table
from ticketrag.ingest import apply_batch
from ticketrag.embed import Embedder
from ticketrag.retrieve import PatternRetriever, pattern_text

STATE = Path('data/processed')
NEXT = Path('dataset/next_days.csv')
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 90)

## 1. Schema and basic shape

In [ ]:
raw = pd.read_csv(NEXT, dtype=str)
REQUIRED = ['unique_key', 'created_date', 'closed_date', 'complaint_type', 'descriptor', 'descriptor_2', 'resolution_description']
missing = [c for c in REQUIRED if c not in raw.columns]
print('shape:', raw.shape)
print('missing required columns:', missing)
print('extra columns (dropped by clean_tickets):', len(set(raw.columns) - set(REQUIRED)))
if 'status' in raw.columns:
    display(raw['status'].value_counts())
created = pd.to_datetime(raw['created_date'], errors='coerce')
print('date range:', created.min(), '->', created.max(), '| unparseable dates:', int(created.isna().sum()))
display(created.dt.date.value_counts().sort_index())

## 2. The pipeline's own cleaner (closed + real resolution + usable labels)

In [ ]:
clean_new, funnel = clean_tickets(raw)
print('cleaning funnel:', funnel)
moji = int(raw['resolution_description'].fillna('').str.contains('â|Ã').sum())
print('rows with double-encoded text (repaired by clean_tickets):', moji)
print('duplicate unique_key inside the batch:', int(raw['unique_key'].duplicated().sum()))
print('columns kept:', list(clean_new.columns))

## 3. Relationship to the training state (leakage check)

In [ ]:
train = pd.read_parquet(STATE / 'tickets.parquet')
pat_old = pd.read_parquet(STATE / 'patterns.parquet')
res_old = pd.read_parquet(STATE / 'resolutions.parquet')
overlap = int(clean_new['unique_key'].isin(set(train['unique_key'])).sum())
print('training range:', train['created_date'].min(), '->', train['created_date'].max())
print('new batch range:', clean_new['created_date'].min(), '->', clean_new['created_date'].max())
print('tickets already in the training state (should be 0):', overlap)
print('training tickets / patterns / resolution templates:', len(train), len(pat_old), len(res_old))

## 4. Coverage: known patterns vs new ticket classes

In [ ]:
known = set(pat_old['pattern_id'])
in_known = clean_new['pattern_id'].isin(known)
print('share of new tickets that fall in an already-known pattern: {:.2%}'.format(in_known.mean()))
new_pat = clean_new[~in_known].groupby(['complaint_type', 'descriptor', 'descriptor_2']).size().sort_values(ascending=False)
print('new patterns:', len(new_pat), '| tickets in them:', int(new_pat.sum()))
display(new_pat.head(15))
new_types = sorted(set(clean_new['complaint_type']) - set(train['complaint_type']))
print('complaint_types never seen in training:', new_types)
res_known = set(res_old['resolution_id'])
unseen = clean_new[~clean_new['resolution_id'].isin(res_known)]
print('tickets with a resolution template never seen:', len(unseen), '| distinct new templates:', unseen['resolution_id'].nunique())
display(unseen['resolution_description'].value_counts().head(5))

## 5. Drift in the biggest known patterns (total-variation distance of the resolution distribution)

0 = identical to training, 1 = completely different. Noise at small n is expected; this shows whether refreshing statistics matters.

In [ ]:
def dist(df):
    return df['resolution_id'].value_counts(normalize=True)

indexed = pat_old.set_index('pattern_id')
rows = []
for pid in clean_new[in_known]['pattern_id'].value_counts().head(15).index:
    a = dist(train[train['pattern_id'] == pid])
    b = dist(clean_new[clean_new['pattern_id'] == pid])
    keys = a.index.union(b.index)
    tv = 0.5 * sum(abs(a.get(k, 0) - b.get(k, 0)) for k in keys)
    r = indexed.loc[pid]
    rows.append({'pattern': pattern_text(r['complaint_type'], r['descriptor'], r['descriptor_2']),
                 'train_n': int((train['pattern_id'] == pid).sum()),
                 'new_n': int((clean_new['pattern_id'] == pid).sum()),
                 'TV_distance': round(float(tv), 3)})
display(pd.DataFrame(rows))

## 6. Effect on the pattern table and tiers (in memory only, nothing written)

In [ ]:
all_t = pd.concat([train, clean_new], ignore_index=True)
pat_new, res_new = build_pattern_table(all_t)
m = pat_old.set_index('pattern_id')[['tier']].join(pat_new.set_index('pattern_id')[['tier']], rsuffix='_after', how='outer')
print('tier transitions (rows = before, columns = after):')
display(pd.crosstab(m['tier'].fillna('absent'), m['tier_after'].fillna('absent')))
print('patterns before/after:', len(pat_old), len(pat_new), '| resolution templates before/after:', len(res_old), len(res_new))

## 7. Dry-run of the real ingestion on a scratch copy of the state

`llm=None`, so new classes are indexed by label only (no API calls). In production the LLM writes example complaints for new classes first.

In [ ]:
tmp = Path(tempfile.mkdtemp(prefix='state_copy_'))
scratch = tmp / 'state'
shutil.copytree(STATE, scratch, ignore=shutil.ignore_patterns('index_*', '*.jsonl'))
emb = Embedder()
rep = apply_batch(scratch, raw, emb, llm=None)
print(json.dumps({k: v for k, v in rep.items() if k != 'filter'}, indent=1, default=str))
rep2 = apply_batch(scratch, raw, emb, llm=None)
print('second ingest of the same batch: ingested =', rep2['ingested'], '| duplicates_skipped =', rep2['duplicates_skipped'])

## 8. Are new classes searchable right after ingestion?

Each new pattern is queried with its own label text. With example complaints (production) retrieval of natural language is better still; this only proves the new vectors are in the index and the retriever sees the new table.

In [ ]:
retr = PatternRetriever(scratch / 'index', scratch, embedder=emb)
newp = pd.read_parquet(scratch / 'patterns.parquet')
added = newp[~newp['pattern_id'].isin(set(pat_old['pattern_id']))].sort_values('total_cases', ascending=False).head(10)
found = []
for r in added.itertuples():
    q = pattern_text(r.complaint_type, r.descriptor, r.descriptor_2)
    top = retr.search(q, k=1)[0]
    found.append(top.pattern_id == r.pattern_id)
    print(('OK  ' if found[-1] else 'MISS'), q, '(n=%d)' % r.total_cases)
print('new patterns retrievable by their own label: %d/%d' % (sum(found), len(found)))

## 9. Verdict

In [ ]:
newp_all = pd.read_parquet(scratch / 'patterns.parquet')
checks = {
    'required columns present': not missing,
    'rows survive the cleaner (> 0)': len(clean_new) > 0,
    'no unique_key overlap with the training state': overlap == 0,
    'ingestion ran on a scratch copy': rep['ingested'] > 0,
    'ingested == usable - duplicates': rep['ingested'] == rep['usable'] - rep['duplicates_skipped'],
    're-ingesting the same batch is a no-op': rep2['ingested'] == 0,
    'pattern count grew by exactly the number of new patterns': len(newp_all) == len(pat_old) + len(rep['new_patterns']),
    'new patterns are retrievable': (len(found) == 0) or all(found),
}
for name, ok in checks.items():
    print('PASS' if ok else 'FAIL', '-', name)
print()
print('OVERALL:', 'next_days can be ingested by the current pipeline' if all(checks.values()) else 'NOT ready, see FAIL lines above')
shutil.rmtree(tmp, ignore_errors=True)  # scratch copy removed; data/processed was never modified